# Standard Logging in MPI Codes

The use of {{poptus}} loggers in MPI-based applications is not much different from their use in serial applications.  This example presents the MPI application called in the User's version of this example to show how its logging output is generated.  Please review that example first.

In [1]:
# Remove input/outputs in rendering

import subprocess as sbp

from pathlib import Path
from IPython.display import Code

import poptus

N_MPI_PROCESSES = 4
MPI_EXEC = Path.cwd().joinpath("example_mpi_program.py")

In [2]:
# Remove input in rendering

Code(filename=Path.cwd().joinpath("my_mpi_method.py"), language='python')

import poptus


def my_mpi_method(logger, mpi_comm, lead_process):
    log, log_debug, warn, log_and_abort = \
        poptus.create_log_functions(logger, "MpiTest")
    
    # Standard MPI information
    comm_size = mpi_comm.Get_size()
    rank = mpi_comm.Get_rank()

    # Log general example information.  These will only be logged by the lead
    # logger.
    log(f"Running MPI example with {comm_size} processes")
    log(f"The lead logging process is rank {rank}")
    log(f"Logging at verbosity level {logger.level}")

    # Simple MPI execution
    log("Starting computations")
    if rank == lead_process:
        # For this method, it's best if the lead log process is also the lead
        # process so that the logging output is simple and clear.  If this lead
        # process block were to call log() directly, that relationship would be
        # necessary.  Therefore, an improved implementation of this method might
        # be to accept a logger configuration and create the logger object
        # itself to automatically set the lead logger as needed.  The present
        # implementation is maintained so that this code is compatible with the
        # Users' version of this example.
        assert logger.is_lead_logger

        workers = [i for i in range(comm_size) if i != lead_process]
        tasks_all = [1.1 * i for i in range(len(workers))]

        for i, task in zip(workers, tasks_all):
            log_debug(f"Sending {task} task to the rank {i} worker process",
                      poptus.LOG_LEVEL_MIN_DEBUG)
            mpi_comm.send(task, dest=i, tag=1)
    else:
        # There is no point to calling log() in this worker process block.
        task = mpi_comm.recv(source=lead_process, tag=1)
        log_debug(f"Received task {task} from lead process",
                  poptus.LOG_LEVEL_MIN_DEBUG)

        if task == 1.1:
            warn("Are you sure that's the task I should work on?")

    mpi_comm.barrier()
    log("Computations finished successfully")

If this function is called within an MPI environment and provided with valid MPI-aware loggers configured to print debug information, the log output would be similar to the following.

In [3]:
# Remove input in rendering

# Run command line program that calls my_mpi_method with nontrivial MPI setup to generate correct log outputs.
CMD = ["mpirun", "-np", str(N_MPI_PROCESSES),
       "python", "-m", "mpi4py",
       MPI_EXEC, str(poptus.LOG_LEVEL_MIN_DEBUG)]
result = sbp.run(CMD, check=True)
assert result.returncode == 0

[MpiTest] Running MPI example with 4 processes
[MpiTest] The lead logging process is rank 0
[MpiTest] Logging at verbosity level 2
[MpiTest] Starting computations
[MpiTest] Sending 0.0 task to the rank 1 worker process
[MpiTest] Sending 1.1 task to the rank 2 worker process
[MpiTest] Sending 2.2 task to the rank 3 worker process
[Rank 1] Received task 0.0 from lead process
[Rank 2] Received task 1.1 from lead process
[Rank 2] WARNING - Are you sure that's the task I should work on?
[Rank 3] Received task 2.2 from lead process
[MpiTest] Computations finished successfully
